# Lab 1: Embedding-Modelle evaluieren und reproduzierbar tracken

Die Demo zeigt live, wie aus einer Zahl im Notebook eine belastbare Messung wird: erst Ähnlichkeit und ein
fragwürdiger Schnelltest, dann Batch- und Echtzeit-Pfad, dann ein Data Contract an einer neuen Datenlieferung, zum
Schluss eine saubere Bewertung als MLflow-Run. Danach folgen drei kurze Übungen, die auf den Ergebnissen der Demo
aufbauen.

Bezug: Kapitel 1 "Von der Entwicklung zum produktiven Einsatz", Kapitel 2 "Architekturgrundlagen für ML-Systeme",
Kapitel 3 "Datenflüsse und Feature-Bereitstellung".

Mitlaufen: `lab1_embeddings_student.ipynb` (ohne Ausgaben, ohne die Absätze "Was wir sehen" und ohne die Lösungen
der Übungen, damit die Fragen an die Gruppe offen bleiben).

**Voraussetzung:** Kapitel 1 bis 3 gehört, vor allem die Folien "Embedding-Modell: ein Satz wird ein Vektor",
"Suchqualität messen: nDCG@10", "Batch-Pfad: Dauer der Indexierung" und "Data Contract prüfen". Daten und Modelle
liegen unter `labs/data/banking77/` und `labs/models/` (einmal `python prefetch.py` mit Netzzugang); danach läuft die
Demo ohne Netzzugriff.

**Ausgangslage:** Die Bank will zu jeder neuen Kundenanfrage frühere Anfragen mit demselben Anliegen finden, damit das
Service-Team schon gegebene Antworten wiederverwenden kann. Bevor ein Embedding-Modell dafür ausgewählt wird, muss
klar sein, wie eine Messung aussieht, auf die sich eine solche Entscheidung stützen kann.

**Ablauf** (Demo rund 25 Minuten, Übungen 25 bis 35 Minuten): Setup (unter 1 min) · Demo 1 Ein Satz wird ein Vektor
(3 min) · Demo 2 Der Schnelltest der Kollegin (3 min) · Demo 3 Batch-Pfad und Echtzeit-Pfad (5 min) · Demo 4 Ein Data
Contract an einer neuen Lieferung (8 min) · Demo 5 Sauber bewerten und festhalten (6 min) · Übung 1 Woher kommt die
0,3? (10 min) · Übung 2 Ist bge-small belegbar besser? (15 min) · Übung 3 (optional) Wer wird Champion? (10 min).

**Bedienung.** In Demo 1 die Variable `ANFRAGE_AUS_DEM_RAUM` mit einer englischen Anfrage aus der Gruppe belegen. Vor
jeder **Frage an die Gruppe** erst die Antworten sammeln, dann die Zelle darunter ausführen. Der Regler in Demo 4
verschiebt nur die Schwelle der Sprachprobe; die Anteile bekannter Wörter sind schon gerechnet, gezählt wird neu. Für
Demo 5 die MLflow-Oberfläche im Ordner `labs/` starten: `mlflow ui --backend-store-uri sqlite:///mlflow.db`. Die
Absätze "Was wir sehen" stehen nur in der Trainerfassung.

In [ ]:
# Ordner labs/ finden und zum Arbeitsordner machen: Das Paket common und die Daten liegen dort.
# Nötig, wenn der Kernel woanders startet (PyCharm, VS Code mit Arbeitsordner im Pack-Root, Quelle in labs/_src/).
import os, sys
from pathlib import Path
for _p in [Path.cwd(), Path.cwd() / "labs", *Path.cwd().parents]:
    if (_p / "common" / "__init__.py").exists():
        os.chdir(_p)
        sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError("Ordner labs/ nicht gefunden. Öffnen Sie das Notebook aus dem Ordner labs/ des Kursmaterials.")

import json
import logging
import time
import warnings

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import mlflow
import numpy as np
import pandas as pd

from common.paths import ARTIFACTS_DIR, offline

offline()  # vor dem ersten Laden eines Modells: die Labs laufen ohne Netzzugriff

from common.contract import GERMAN_QUERIES, NEW_DELIVERY, check_contract, known_word_share
from common.data import dataset_hash, load_banking77
from common.encoder_eval import (corpus_embeddings, hits_table, ndcg_per_query, package_versions,
                                 paired_bootstrap, query_embeddings)
from common.evalset import build_evalset
from common.interactive import explore
from common.metrics import retrieval_metrics, top_k
from common.models import EMBEDDING_MODELS, encode, load_encoder, size_mb, timed_encode
from common.plotstyle import use_course_style
from common.tracking import setup_mlflow

use_course_style()
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)  # Hinweis beim Laden von rohem DistilBERT
SEED, DEVICE, BATCH_SIZE = 1, "cpu", 64
OUT_DIR = ARTIFACTS_DIR / "lab1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

logging.disable(logging.INFO)  # das erste Anlegen der Datenbank schreibt sonst viele Migrationszeilen
tracking_uri = setup_mlflow("lab1")
logging.disable(logging.NOTSET)
logging.getLogger("mlflow").setLevel(logging.ERROR)

train, test, intents = load_banking77()
print(f"{len(train)} Trainings- und {len(test)} Testanfragen, {len(intents)} Anliegen")

## Demo 1: Ein Satz wird ein Vektor (Kapitel 1)

Drei echte Anfragen aus Banking77: zwei zum Anliegen `card_arrival`, eine zu `exchange_rate`. Wir betten sie mit
`minilm-l6` (91 MB, auf Satzpaaren trainiert) und mit rohem DistilBERT ein (265 MB, nur Mean-Pooling, kein
Satz-Training) und vergleichen die Cosine-Similarity. Kapitel 1 hat den Unterschied schon gezeigt; hier sieht die
Gruppe ihn am eigenen Satz (`ANFRAGE_AUS_DEM_RAUM`).

In [ ]:
minilm = load_encoder("minilm-l6", device=DEVICE)
distilbert = load_encoder("distilbert-raw", device=DEVICE)

a = "I am still waiting on my card?"   # card_arrival
b = "My card isn't here yet."          # card_arrival
c = "What exchange rate is it?"        # exchange_rate
ANFRAGE_AUS_DEM_RAUM = "My new card has still not arrived"  # hier eine Anfrage aus der Gruppe eintragen

table = {}
for name, model in [("minilm-l6", minilm), ("distilbert-raw", distilbert)]:
    emb = encode(model, [a, b, c, ANFRAGE_AUS_DEM_RAUM])
    sim = emb @ emb.T  # normierte Vektoren: Skalarprodukt = Cosine
    table[name] = {
        "a, b (gleiches Anliegen)": sim[0, 1],
        "a, c (anderes Anliegen)": sim[0, 2],
        "Abstand": sim[0, 1] - sim[0, 2],
        "Raum, a": sim[3, 0],
        "Raum, c": sim[3, 2],
    }
cosines = pd.DataFrame(table)
print(f"Größe: minilm-l6 {size_mb(minilm):.0f} MB, distilbert-raw {size_mb(distilbert):.0f} MB")
cosines.round(3)

## Demo 2: Der Schnelltest der Kollegin (Kapitel 1)

Eine Kollegin hat MiniLM letzte Woche kurz getestet: Zu jeder Testanfrage sucht sie die ähnlichste
Trainingsanfrage und zählt, wie oft deren Anliegen stimmt. Das ist Hit@1. Ihre Zelle:

In [ ]:
# Schnelltest MiniLM
import pandas as pd
from sentence_transformers import SentenceTransformer

train_df = pd.read_parquet("data/banking77/train.parquet")
test_df = pd.read_parquet("data/banking77/test.parquet")
model = SentenceTransformer("models/minilm-l6", device="cpu")

corpus = train_df.iloc[:3000]  # Teilmenge, damit es schnell geht
queries = test_df.sample(200)
corpus_emb = model.encode(corpus["text"].tolist(), normalize_embeddings=True)
query_emb = model.encode(queries["text"].tolist(), normalize_embeddings=True)

best = (query_emb @ corpus_emb.T).argmax(axis=1)
hit_rate = (corpus["label"].to_numpy()[best] == queries["label"].to_numpy()).mean()
print(f"Trefferquote Top-1: {hit_rate:.2f}")

**Frage an die Gruppe:** In der Messtabelle von Kapitel 3 hatte MiniLM Hit@1 0,927. Die Kollegin misst rund 0,3 und
schlägt vor, MiniLM auszusortieren. Welcher Zahl trauen Sie, und was müssten Sie wissen, um es zu entscheiden?
(Zelle ruhig ein zweites Mal ausführen.)

Die nächste Zelle rechnet den Test der Kollegin 20-mal, jedes Mal mit einer anderen Stichprobe von 200 Anfragen
(Seeds 0 bis 19). Korpus und Modell bleiben gleich.

In [ ]:
test_emb = model.encode(test_df["text"].tolist(), normalize_embeddings=True)  # alle Testanfragen einmal
first_ok = corpus["label"].to_numpy()[(test_emb @ corpus_emb.T).argmax(axis=1)] == test_df["label"].to_numpy()
runs = [first_ok[np.random.default_rng(s).choice(len(test_df), size=200, replace=False)].mean() for s in range(20)]
print(f"20 Läufe mit je 200 Anfragen: Trefferquote zwischen {min(runs):.2f} und {max(runs):.2f}, "
      f"im Mittel {np.mean(runs):.2f}")

## Demo 3: Batch-Pfad und Echtzeit-Pfad (Kapitel 2)

Die Suche hat zwei Pfade. Im **Batch-Pfad** werden alle früheren Anfragen einmal eingebettet und in den Index
geschrieben; dort zählt der Durchsatz. Im **Echtzeit-Pfad** kommt eine einzelne Anfrage an; dort zählt die Latenz.
Die Folien in Kapitel 2 zeigen beide Messungen; hier laufen sie auf dem Rechner im Raum.

In [ ]:
index_emb, seconds = timed_encode(minilm, train["text"].tolist(), BATCH_SIZE)  # Batch-Pfad: der ganze Korpus
rate = len(index_emb) / seconds
hours_10m = 10_000_000 / rate / 3600

print(f"Index: {index_emb.shape[0]} Vektoren mit {index_emb.shape[1]} Dimensionen in {seconds:.1f} s")
print(f"Durchsatz: {rate:,.0f} Texte je Sekunde".replace(",", " "))
print(f"Hochrechnung auf 10 Mio. Dokumente: {hours_10m:.1f} h auf diesem Rechner")

In [ ]:
# Echtzeit-Pfad: eine Anfrage einbetten und im Index suchen, 50-mal gemessen
samples = test["text"].sample(50, random_state=SEED).tolist()
encode(minilm, samples[:5], batch_size=1)  # Warmlauf
encode_ms, search_ms = [], []
for text in samples:
    t0 = time.perf_counter()
    vector = encode(minilm, [text], batch_size=1)
    t1 = time.perf_counter()
    hits = top_k(vector, index_emb, k=10)
    search_ms.append((time.perf_counter() - t1) * 1e3)
    encode_ms.append((t1 - t0) * 1e3)
print(f"Encoder je Anfrage: Median {np.median(encode_ms):.1f} ms "
      f"(aus dem Durchsatz gerechnet wären es {1e3 / rate:.2f} ms je Text)")
print(f"Suche über {len(index_emb)} Vektoren: Median {np.median(search_ms):.2f} ms")
print(f"Letzte Anfrage: {text!r} -> Treffer 1: {train['text'].iloc[hits[0, 0]]!r}")

**Frage an die Gruppe:** Die Suche ist hier ein Skalarprodukt mit der ganzen Index-Matrix. Bei 10 000 Dokumenten kostet
sie einen Bruchteil einer Millisekunde. Was kostet sie je Anfrage bei 10 Millionen Dokumenten, und wie viel
Arbeitsspeicher braucht der Index dann?

Die Zelle vervielfacht den Index künstlich (dieselben Vektoren 1-, 5- und 20-mal) und misst die Suche. Sie misst nur
die Suche, ohne Encoder-Aufruf dazwischen wie im Anfragepfad; der Wert für den einfachen Index liegt deshalb meist
etwas unter dem von oben. Für die Hochrechnung zählt die Steigung.

In [ ]:
probe_vectors = encode(minilm, samples[:20], batch_size=1)
search_rows = []
for factor in [1, 5, 20]:
    big_index = np.tile(index_emb, (factor, 1))
    times = []
    for vector in probe_vectors:
        t0 = time.perf_counter()
        top_k(vector[None], big_index, k=10)
        times.append((time.perf_counter() - t0) * 1e3)
    search_rows.append({"Dokumente": len(big_index), "MB": big_index.nbytes / 1e6, "Suche ms (Median)": np.median(times)})
del big_index
scaling = pd.DataFrame(search_rows)
ms_10m = scaling["Suche ms (Median)"].iloc[-1] / scaling["Dokumente"].iloc[-1] * 10_000_000
print(f"Hochrechnung auf 10 Mio. Dokumente: rund {ms_10m:.0f} ms je Anfrage, "
      f"{10_000_000 * index_emb.shape[1] * 4 / 1e9:.1f} GB Index")
scaling.round(2)

## Demo 4: Ein Data Contract an einer neuen Lieferung (Kapitel 3)

Auf der Folie "Data Contract prüfen" verletzt keine der 13 069 Anfragen von Banking77 den Contract. Nur die
ASCII-Probe meldet 61 Anfragen, vor allem mit £ und €; deshalb erlaubt der Contract `encoding: UTF-8`. Banking77 ist
aber ein bereinigter Datensatz. Jetzt kommt eine neue Lieferung des Kundenservice mit acht Zeilen, wie sie im Betrieb
vorkommen (von Hand geschrieben):

In [ ]:
pd.DataFrame({"Anliegen": NEW_DELIVERY["intent"], "Zeichen": NEW_DELIVERY["text"].str.len(),
              "Text (gekürzt)": NEW_DELIVERY["text"].str.slice(0, 60)}).set_index(NEW_DELIVERY["fall"])

**Frage an die Gruppe:** Welche dieser acht Zeilen hält der Contract von der Folie auf?

Die nächste Zelle prüft die Regeln der Folie und eine sechste Spalte, die wir danach besprechen.

In [ ]:
real = pd.concat([train, test], ignore_index=True)
print("Verstöße in den 13 069 echten Anfragen:", check_contract(real, intents).sum().to_dict())

violations = check_contract(NEW_DELIVERY, intents)
slide_rules = violations.columns[:5]  # die fünf Regeln der Folie; die sechste ist die Variante ohne Trennzeichen
pd.DataFrame({
    "Regeln der Folie verletzt": violations[slide_rules].apply(lambda row: ", ".join(slide_rules[row]) or "-", axis=1),
    "Ziffern ohne Trennzeichen": violations.iloc[:, 5].map({True: "x", False: ""}),
}).set_index(NEW_DELIVERY["fall"])

In [ ]:
contract_rules = [rule for rule in slide_rules if rule != "Zeichen außerhalb ASCII"]  # encoding: UTF-8 im Contract
problem_rows = NEW_DELIVERY["fall"] != "normale Anfrage"
print(f"Von {problem_rows.sum()} Problemzeilen aufgehalten: {violations[contract_rules].any(axis=1).sum()} nach den "
      f"Regeln des Contracts, {violations[slide_rules].any(axis=1).sum()} mit der ASCII-Probe der Folie")

Der Contract verlangt `language: en`. Was kostet eine deutsche Anfrage im englischen Index? 20 Anfragen aus dem
Eval-Set, von Hand übersetzt, gegen den englischen Index aus Demo 3:

In [ ]:
german_labels = np.array([intents.index(intent) for _, intent, _ in GERMAN_QUERIES])
language_metrics = {}
for language, column in [("englisch (Original)", 0), ("deutsch (übersetzt)", 2)]:
    vectors = encode(minilm, [row[column] for row in GERMAN_QUERIES])
    m = language_metrics[language] = retrieval_metrics(vectors, german_labels, index_emb, train["label"].to_numpy())
    print(f"{language:20s} Hit@1 {m['hit_at_1']:.2f}   nDCG@10 {m['ndcg_at_10']:.3f}   keine Fehlermeldung")

unk = sum(minilm.tokenizer.unk_token_id in ids for ids in minilm.tokenizer([g for _, _, g in GERMAN_QUERIES])["input_ids"])
print(f"Deutsche Anfragen mit [UNK]-Token: {unk} von {len(GERMAN_QUERIES)} (der Tokenizer zerlegt in Wortstücke)")

Eine Sprachprobe ohne Zusatzpaket: Welcher Anteil der Wörter steht im Vokabular des Tokenizers als ganzes Wort?
Deutsche Wörter zerfallen in Wortstücke. Unterhalb der Schwelle gilt eine Anfrage als nicht englisch. Am Regler: Wo
verliert die Probe deutsche Anfragen, wo meldet sie echte englische?

In [ ]:
share_real = known_word_share(real["text"].tolist(), minilm.tokenizer)
share_german = known_word_share([g for _, _, g in GERMAN_QUERIES], minilm.tokenizer)
share_delivery = known_word_share(NEW_DELIVERY["text"].tolist(), minilm.tokenizer)
print(f"Anteil bekannter Wörter: echte Anfragen Median {np.median(share_real):.2f} ({np.mean(share_real == 1):.0%} "
      f"genau 1,0), deutsche Übersetzungen Median {np.median(share_german):.2f}")


def language_probe(schwelle):
    """Meldet Anfragen, deren Anteil bekannter Wörter unter der Schwelle liegt."""
    flagged = share_real < schwelle
    print(f"Schwelle {schwelle:.2f}: deutsche Anfragen erkannt {np.sum(share_german < schwelle)} von "
          f"{len(share_german)}, Fehlalarme {flagged.sum()} von {len(real)} echten englischen Anfragen")
    print("Lieferung gemeldet:", list(NEW_DELIVERY["fall"][share_delivery < schwelle]))
    for text in real["text"][flagged].head(3):
        print("  Fehlalarm:", text)


explore(language_probe, schwelle=(0.6, 0.3, 0.9, 0.05))

In [ ]:
probe = pd.DataFrame({f"{s:.2f}".replace(".", ","): {"deutsch erkannt (von 20)": int(np.sum(share_german < s)),
                                                     "Fehlalarme (von 13 069)": int(np.sum(share_real < s))}
                      for s in [0.4, 0.5, 0.6, 0.65, 0.7, 0.8]})
probe

## Demo 5: Sauber bewerten und festhalten (Kapitel 1 bis 3)

Jetzt dieselbe Frage wie im Schnelltest, aber so, dass jemand die Zahl in vier Wochen nachrechnen kann. Drei Dinge
machen das möglich: ein **Daten-Hash** (welche Daten), ein **Eval-Set mit Fingerabdruck** (welche Queries, welcher
Korpus, fester Seed) und ein **Run** in MLflow, der Einstellungen, Ergebnis und Versionen zusammenhält.

In [ ]:
DATA_HASH = dataset_hash(train, test)[:16]
evalset = build_evalset(train, test, n_queries=1000, seed=SEED)
assert evalset.corpus == train["text"].tolist()  # Korpus des Eval-Sets = Index aus Demo 3
print(f"Daten-Hash {DATA_HASH}, Eval-Set {evalset.fingerprint} (Seed {evalset.seed}): "
      f"{len(evalset.queries)} Queries gegen {len(evalset.corpus)} Dokumente")
assert DATA_HASH == "3176e3ac9aef7185" and evalset.fingerprint == "e21d87d6477c7f22"

encode(minilm, evalset.queries[:BATCH_SIZE], BATCH_SIZE)  # Warmlauf
query_emb, query_seconds = timed_encode(minilm, evalset.queries, BATCH_SIZE)
metrics = retrieval_metrics(query_emb, evalset.query_labels, index_emb, evalset.corpus_labels, k=10)
metrics["texts_per_second"] = len(evalset.queries) / query_seconds
metrics["size_mb"] = size_mb(minilm)
pd.Series(metrics).round(3)

Der Mittelwert allein zeigt nicht, wie sich die Fehler verteilen. Eine Tabelle je Anfrage zeigt es, und sie wird das
Artefakt des Runs.

**Frage an die Gruppe:** Bei Hit@1 0,927 liegt der erste Treffer bei 73 von 1 000 Anfragen daneben. Verteilen sich die
Fehler gleichmäßig über die 77 Anliegen, oder häufen sie sich? Und woran könnte es liegen?

In [ ]:
hits = hits_table(evalset, intents, query_emb, index_emb)
wrong = hits[hits["hit_at_1"] == 0]
errors = wrong["intent"].value_counts()
print(f"Erster Treffer falsch: {len(wrong)} von {len(hits)} Anfragen, verteilt auf {len(errors)} der {len(intents)} "
      f"Anliegen; die zehn fehlerreichsten tragen {errors.head(10).sum()} davon")
print(f"Ohne einen passenden Treffer unter den Top 10 (nDCG@10 = 0): {(hits['ndcg_at_10'] == 0).sum()} Anfragen")
per_intent = hits.groupby("intent").agg(anfragen=("query", "size"), ndcg_at_10=("ndcg_at_10", "mean"))
per_intent["falscher Treffer 1 (Anzahl)"] = wrong.groupby("intent")["top1_intent"].agg(
    lambda s: ", ".join(f"{name} ({n})" for name, n in s.value_counts().head(2).items()))
print(f"Anfragen je Anliegen im Eval-Set: {per_intent['anfragen'].min()} bis {per_intent['anfragen'].max()}")
per_intent.sort_values("ndcg_at_10").head(5).round(3)

Zum Schluss der Run: Daten-Hash, Fingerabdruck, Einstellungen, Paketversionen und die Tabelle je Anfrage hängen an
einem Eintrag in MLflow.

In [ ]:
with mlflow.start_run(run_name="minilm-l6-baseline") as run:
    mlflow.log_params({
        "model": "minilm-l6", "hub_id": EMBEDDING_MODELS["minilm-l6"],
        "n_queries": len(evalset.queries), "n_corpus": len(evalset.corpus), "k": 10,
        "batch_size": BATCH_SIZE, "device": DEVICE,
        "max_seq_length": minilm.max_seq_length, "pooling": minilm[1].pooling_mode, "normalize": True,
    })
    mlflow.log_metrics(metrics)
    mlflow.set_tags({
        "dataset_hash": DATA_HASH, "evalset_fingerprint": evalset.fingerprint, "evalset_seed": SEED,
        **{f"version.{package}": version for package, version in package_versions().items()},
    })
    hits_path = OUT_DIR / "treffer_je_anfrage_minilm-l6.csv"
    hits.to_csv(hits_path, index=False)
    mlflow.log_artifact(str(hits_path), artifact_path="analysis")

stored = mlflow.get_run(run.info.run_id)  # aus der Datenbank zurückgelesen, nicht aus dem Notebook
print(f"Run {stored.info.run_id}: {len(stored.data.params)} Parameter, {len(stored.data.metrics)} Metriken, "
      f"{len(stored.data.tags)} Tags")
print("Tags:", {key: value for key, value in stored.data.tags.items() if not key.startswith("mlflow.")})

In der MLflow-Oberfläche (Experiment `lab1`) stehen am Run die Einstellungen als Parameter, Daten-Hash,
Fingerabdruck und Paketversionen als Tags, die Tabelle je Anfrage unter `analysis/`. Start im Ordner `labs/`:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [ ]:
summary = {
    "lab": "lab1",
    "dataset_hash": DATA_HASH,
    "evalset_fingerprint": evalset.fingerprint,
    "cosine": {model: {key: round(float(value), 3) for key, value in col.items()} for model, col in cosines.items()},
    "quick_test_hit_rate_unseeded": round(float(hit_rate), 3),
    "quick_test_20_runs": {"min": round(float(min(runs)), 3), "max": round(float(max(runs)), 3),
                           "mean": round(float(np.mean(runs)), 3)},
    "indexing": {"texts_per_second": round(rate), "hours_for_10m": round(hours_10m, 2),
                 "ms_per_text_in_batch": round(1e3 / rate, 3)},
    "query_path_ms": {"encode_median": round(float(np.median(encode_ms)), 2),
                      "search_median": round(float(np.median(search_ms)), 3),
                      "search_10m_extrapolated": round(float(ms_10m))},
    "contract": {
        "real_violations": {k: int(v) for k, v in check_contract(real, intents).sum().items()},
        "delivery_caught_by_slide_rules": int(violations[slide_rules].any(axis=1).sum()),
        "delivery_caught_by_contract_rules": int(violations[contract_rules].any(axis=1).sum()),
        "delivery_rows": len(NEW_DELIVERY),
        "german_20": {language: {k: round(v, 3) for k, v in m.items()} for language, m in language_metrics.items()},
        "german_with_unk": int(unk),
        "language_probe": {col: {k: int(v) for k, v in probe[col].items()} for col in probe.columns},
    },
    "minilm_l6": {key: round(float(value), 4) for key, value in metrics.items()},
    "zero_ndcg_queries": int((hits["ndcg_at_10"] == 0).sum()),
    "mlflow_run_id": run.info.run_id,
}
(OUT_DIR / "results.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Geschrieben:", sorted(path.name for path in OUT_DIR.iterdir()))

## Übungen

Drei kurze Übungen, zusammen rund 35 Minuten (Übung 3 optional). Sie bauen auf den Variablen der Demo auf: Führen Sie
vorher alle Zellen der Demo einmal aus. Code schreiben Sie an der Stelle mit "Auftrag", die Zeilen darunter prüfen
und zeigen das Ergebnis.

### Übung 1: Woher kommt die 0,3?

Der Schnelltest der Kollegin (Demo 2) meldet für MiniLM rund 0,3, die saubere Messung in Demo 5 Hit@1 0,927. Beide
Zahlen messen dasselbe: den Anteil der Anfragen, deren erster Treffer das richtige Anliegen hat. Finden Sie die
Ursache der Lücke und belegen Sie sie mit Zahlen aus den Daten. Zur Hand sind `corpus` (die 3 000 Trainingsanfragen
der Kollegin), `test_df`, `first_ok` (Treffer auf Platz 1 je Testanfrage in diesem Korpus, Demo 2), `test_emb` (alle
Testanfragen) und `index_emb` (der ganze Korpus, Demo 3). **Darf MiniLM aussortiert werden?** (rund 10 Minuten)

In [ ]:
# Auftrag: Ursache der Lücke finden und mit Zahlen aus den Daten belegen.
# Danach existieren: ursache (ein Satz, Text) und beleg (dict: Beschreibung -> Zahl, mindestens ein Eintrag)
# TODO: Ihre Lösung
...

assert isinstance(ursache, str) and isinstance(beleg, dict) and beleg, "ursache (Text) und beleg (dict) fehlen"
print(ursache)
for description, value in beleg.items():
    print(f"  {description}: {value:.3f}" if isinstance(value, float) else f"  {description}: {value}")

<details><summary>Hinweis</summary>

Ein Treffer zählt nur, wenn das richtige Anliegen im Korpus überhaupt vorkommt. Sehen Sie sich `corpus["intent"]` an,
zählen Sie die Anliegen mit `nunique()` und prüfen Sie mit `isin`, wie viele Testanfragen ihr Anliegen dort
wiederfinden.

</details>

**Ihre Antwort:** ...

### Übung 2: Ist bge-small belegbar besser?

`bge-small` erreicht auf dem Eval-Set aus Demo 5 einen höheren nDCG@10 als MiniLM, rechnet aber langsamer (Folie in
Kapitel 2: 1,9 statt 1,1 h für 10 Mio. Dokumente). Ob der Abstand mehr ist als Zufall der 1 000 Anfragen, zeigt ein
gepaarter Bootstrap: 2 000-mal 1 000 Anfragen mit Zurücklegen ziehen und für **dieselben** gezogenen Anfragen die
mittlere Differenz beider Modelle rechnen. Das 2,5-%- und das 97,5-%-Quantil dieser Differenzen begrenzen das
95-%-Intervall; liegt es ganz über 0, ist der Vorsprung belegt. **Ist er groß genug, um dafür das langsamere Modell zu
betreiben?** (rund 15 Minuten)

In [ ]:
# Auftrag:
# 1. per_query: dict Modellname -> nDCG@10 je Anfrage für "minilm-l6" und "bge-small" auf evalset
#    (ndcg_per_query mit query_embeddings und corpus_embeddings; die Vektoren kommen aus labs/.cache/).
# 2. ci: (Differenz bge-small minus minilm-l6, untere Grenze, obere Grenze), gepaarter Bootstrap mit 2 000 Ziehungen
#    aus np.random.default_rng(SEED).
# Danach existieren: per_query, ci
# TODO: Ihre Lösung
...

assert np.allclose(ci, paired_bootstrap(per_query["minilm-l6"], per_query["bge-small"], seed=SEED), atol=0.002), \
    "weicht deutlich vom gepaarten Bootstrap in common.encoder_eval ab"
print(f"nDCG@10 minilm-l6 {per_query['minilm-l6'].mean():.3f}, bge-small {per_query['bge-small'].mean():.3f}; "
      f"Differenz {ci[0]:+.3f}, 95-%-Intervall {ci[1]:+.3f} bis {ci[2]:+.3f}; gleicher nDCG@10 bei "
      f"{np.mean(per_query['bge-small'] == per_query['minilm-l6']):.0%} der Anfragen")

<details><summary>Hinweis</summary>

Gepaart heißt: Eine Ziehung von Anfrage-Indizes gilt für beide Modelle. Am einfachsten ziehen Sie gleich aus der
Differenz je Anfrage. `idx = rng.integers(0, n, size=(2000, n))` liefert alle Ziehungen als Matrix, `diff[idx]` hat
dann die Form `(2000, n)`, der Mittelwert über `axis=1` ist je Wiederholung eine Differenz. Die Grenzen liefert
`np.percentile(..., [2.5, 97.5])`.

</details>

**Ihre Antwort:** ...

### Übung 3 (optional): Wer wird Champion?

Welches Modell in Betrieb geht, entscheidet eine Regel, die vor dem Vergleich feststeht: nDCG@10 mindestens 0,85;
unter den Modellen, die das bestehen, gewinnt das mit dem höchsten nDCG@10. Ist es nicht das schnellste (hier
`minilm-l6`), muss sein Vorsprung belegt sein (untere Grenze aus Übung 2 über 0) und mindestens `min_lead` betragen,
sonst gewinnt das schnellere. **Wie hängt der Champion an `min_lead`, und wer sollte den Wert festlegen?**
(rund 10 Minuten)

In [ ]:
MIN_NDCG, FASTEST = 0.85, "minilm-l6"  # MiniLM rechnet schneller (Kapitel 2)

# Auftrag: choose_champion(min_lead) -> Name des Champions nach der Regel oben, aus per_query und ci (Übung 2).
# Besteht kein Modell die Qualitätsschwelle, soll ein RuntimeError das sagen.
# Danach existiert: choose_champion
# TODO: Ihre Lösung
...

print({f"min_lead {min_lead}": choose_champion(min_lead) for min_lead in (0.0, 0.01, 0.02)})

**Ihre Antwort:** ...

## Dokumentation zum Nachschlagen

Kursversionen: sentence-transformers 5.7, transformers 4.57, mlflow 2.22, pandas 2.3, numpy 2.0. Die Doku von
Sentence Transformers zeigt nur die aktuelle Fassung; die hier genutzten Parameter sind dort unverändert beschrieben.

**Sentence Transformers** (Demo 1, 2, 3 und 5; im Lab-Paket `load_encoder` und `encode`)

- `SentenceTransformer(pfad, device="cpu")`, `encode(texte, batch_size=..., normalize_embeddings=True)`,
  `max_seq_length`: https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html
- Module `Pooling` (`pooling_mode`, im Run als Parameter `pooling`) und `Normalize`:
  https://sbert.net/docs/package_reference/sentence_transformer/models.html
- Modellkarte `minilm-l6` (384 Dimensionen, Abschneiden nach 256 Word Pieces):
  https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

**Tokenizer** (Demo 4, `[UNK]`-Zählung)

- Aufruf des Tokenizers mit Rückgabe `input_ids`, Sondertoken `unk_token` und `unk_token_id`:
  https://huggingface.co/docs/transformers/v4.57.0/en/main_classes/tokenizer

**Suche und Metriken** (Demo 3 und 5; im Lab-Paket `top_k` und `retrieval_metrics`)

- `np.argpartition` für die Top 10 (Reihenfolge innerhalb der Teile undefiniert, deshalb danach sortiert):
  https://numpy.org/doc/2.0/reference/generated/numpy.argpartition.html
- nDCG, MRR und Hit Rate zum Abgleich mit einer Bibliothek (ranx, Variante `ndcg` nach Järvelin):
  https://amenra.github.io/ranx/metrics/

**Daten und Fingerabdruck** (Demo 2 und 5; im Lab-Paket `load_banking77`, `dataset_hash`, `build_evalset`)

- `pd.read_parquet`: https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.read_parquet.html
- `DataFrame.sample(n, random_state=...)`, ohne `random_state` jedes Mal eine andere Stichprobe:
  https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.sample.html
- `np.random.default_rng(seed)` und `Generator.choice(..., replace=False)`:
  https://numpy.org/doc/2.0/reference/random/generator.html
- `hashlib.sha256`, `update`, `hexdigest`: https://docs.python.org/3/library/hashlib.html

**Data Contract** (Demo 4; im Lab-Paket `check_contract`)

- `str.isascii`: https://docs.python.org/3/library/stdtypes.html#str.isascii

**MLflow 2.22** (Demo 5)

- `mlflow.start_run(run_name=...)`, `log_params`, `log_metrics`, `set_tags`, `log_artifact`, `get_run`:
  https://mlflow.org/docs/2.22.0/api_reference/python_api/mlflow.html
- SQLite als Backend Store (die Registry braucht einen Datenbank-Speicher):
  https://mlflow.org/docs/2.22.0/tracking/backend-stores/

**Übungen** (im Lab-Paket `ndcg_per_query`, `paired_bootstrap`, `query_embeddings`, `corpus_embeddings` in
`common/encoder_eval.py`)

- `Series.isin` und `Series.nunique` (Übung 1):
  https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.isin.html,
  https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.nunique.html
- `Generator.integers` für die Ziehungen (Übung 2): https://numpy.org/doc/2.0/reference/random/generator.html
- `np.percentile` (Standard `method="linear"`) für die Grenzen des Intervalls (Übung 2):
  https://numpy.org/doc/2.0/reference/generated/numpy.percentile.html

**Regler** (Demo 4, `explore`)

- `explore` baut auf `ipywidgets.interact` auf; das Tupel `(start, min, max, step)` wird zum Schieberegler mit
  Startwert: https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html